In [30]:
from scipy.spatial import distance
import pandas as pd
import numpy as np
from pymol import cmd
import pymol
import os

In [4]:
# Define some paths
root = '../../../Documents_GPU/mtb-targeted-protein-degradation/notebooks'

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))
# pocket_detection_data_interpro = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data_interpro.tsv"), sep='\t')

PROTEINS = sorted(set(pocket_detection_data['Uniprot AC']))

In [28]:
def get_pockets_to_display(df):
    pockets = {}
    for fn, pn, ps, coord in df[['File name', 'Pocket number', 'Pocket score', 'Pocket centroid coordinate (x y z)']].values:
        label = fn.replace(".pdb", "") + "_pocket_" + str(pn)
        coords = np.array(coord.split(), dtype=float)
        keep = True
        for pocket in pockets:
            if distance.euclidean(coords, pockets[pocket][0]) < 6.14:
                keep = False
                break
        if keep:
            pockets[label] = [coords, ps]
    return pockets

In [42]:
for protein in PROTEINS:

    # Filter pockets
    df = pocket_detection_data[pocket_detection_data['Uniprot AC'] == protein].sort_values('Pocket score', ascending=False)

    # Select reference structure
    ref_st = df['File name'].tolist()[0]

    # Identify pockets to display
    pockets = get_pockets_to_display(df)
    
    # Create pymol session
    pymol.finish_launching(['pymol','-cq'])
    cmd.reinitialize()
    cmd.bg_color("white")
    cmd.set("ray_opaque_background", 0)

    # Load structure
    cmd.load(os.path.join(root, "..", "processed", "aligned_relaxed_structures", protein, ref_st), "structure")
    cmd.set_color("structure_color", [0x8D/255, 0xC7/255, 0xFA/255])
    cmd.color("structure_color", "structure")
    cmd.show('surface')
    cmd.hide('cartoon')
    cmd.set("transparency", 0.3, "structure")

    cmd.save(os.path.join("/home/acomajuncosa/Documents_GPU/tmp/session.pse"))